### 동시출현 행렬
- 특정 단어를 기준으로 주변의 범위 안에서 어떠한 단어가 등장했는가? 빈도수를 확인하는 방법 
- 예
    - '오늘 날씨가 너무 좋다.' , '내일 날씨가 조금 흐리다'
    - 범위(window)가 1인 경우 
    - 날씨가 -> 주변 단어 ( '오늘', '너무', '내일', '조금' )
- N-gram 차이 
    - N-gram : 단어 순서 반영 
    - 동시출현행렬 : 의미 관계를 반영

#### PMI
- 두 단어가 우연히 함께 등장했는가? 아니면 의미적으로 연관성이 있는가? 측정 
- 측정 값이 클수록 의미적으로 강하게 연결되어있다.

#### PPMI
- PMI가 음수인 경우는 극히 드문 경우 
- 음수인 데이터를 0으로 대체하는 값

In [ ]:
import math 
import pandas as pd 
from konlpy.tag import Okt

In [ ]:
docs = [
    '오늘 날씨가 너무 좋다', 
    '오늘 기분이 정말 좋다', 
    '내일 날씨가 조금 흐리다', 
    '기분이 매우 나쁘다'
]

In [ ]:
# 좌우 단어의 검색 구간을 지정 
window_size = 2

In [ ]:
# 분석기를 이용하여 문장을 형태소 별로 분리 
okt = Okt()

tokens = [
    okt.morphs(doc) for doc in docs
]
tokens

In [ ]:
new_list = []
for token in tokens:
    # new_list += token
    # 리스트 포함시킨다. 
    # list.extend() -> 새로운 리스트를 포함시킨다. 
    # extended() -> 리스트를 포함시킨다. 결과를 저장x -> 변수에 대입 
    new_list.extend(token)
    
new_list

In [ ]:
vocab = sorted(set(sum( tokens, [] )))

In [ ]:
# 단어들의 목록을 인덱스와 함께 dict 형태로 저장 
vocab_dict = {
    word : idx for idx, word in enumerate(vocab)
}
vocab_dict

In [ ]:
# 동시 출현 행렬을 만들기 위해서 0행렬을 생성 
co_metric = [ [0] * len(vocab) for i in range(len(vocab)) ]
co_metric

In [ ]:
# window_size를 기반으로 동시출현 카운트를 체크 
for token in tokens:
    # print(token)
    # break
    for idx, word in enumerate(token):
        # print(word)
        # idx : 위치 값
        # word : 단어
        count_idx  = vocab_dict[word]
        # 윈도우의 범위 지정 
        start = max( 0, idx - window_size )
        end = min(len(token), idx + window_size)
        # start, end를 이용하여 반복문을 생성 
        for i in range(start, end):
            if idx != i:            ## 단어의 위치가 같지 않은 경우
                # print(token[i], token[idx])
                context = token[i]
                context_idx = vocab_dict[context]
                # co_metric에 특정 위치에 1를 더해준다. 
                co_metric[count_idx][context_idx] += 1
        
                

In [39]:
co_metric

[[0, 0, 0, 2, 1, 1, 0, 1, 0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0],
 [2, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0],
 [0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
 [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 2, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0],
 [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1],
 [1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]]

In [ ]:
co_df = pd.DataFrame(co_metric, index = vocab, columns = vocab)
co_df

In [ ]:
# PMI 계산 
# 각 단어별 동시 등장 횟수의 합계 / total_count
total_count = sum( sum(row) for row in co_metric )
total_count

In [ ]:
total_count2 = co_df.sum().sum()

In [41]:
p_word = [sum(row) / total_count for row in co_metric]
p_word

[0.14634146341463414,
 0.07317073170731707,
 0.04878048780487805,
 0.0975609756097561,
 0.024390243902439025,
 0.07317073170731707,
 0.07317073170731707,
 0.04878048780487805,
 0.12195121951219512,
 0.07317073170731707,
 0.07317073170731707,
 0.0975609756097561,
 0.04878048780487805]

In [ ]:
len(vocab)

In [47]:
p_context = [
    sum(row) / total_count for row in co_df.T.values
]
p_context

[np.float64(0.14634146341463414),
 np.float64(0.12195121951219512),
 np.float64(0.024390243902439025),
 np.float64(0.14634146341463414),
 np.float64(0.04878048780487805),
 np.float64(0.04878048780487805),
 np.float64(0.04878048780487805),
 np.float64(0.0975609756097561),
 np.float64(0.14634146341463414),
 np.float64(0.04878048780487805),
 np.float64(0.04878048780487805),
 np.float64(0.04878048780487805),
 np.float64(0.024390243902439025)]

In [49]:
def calc_pmi(i, j):
    p_wc = co_metric[i][j] / total_count
    if p_wc == 0:
        result = 0
    else:
        result = math.log2(p_wc / ( p_word[i] * p_context[j] ) + 1e-12 )
    return result

pmi_metric = [ 
                [
                    calc_pmi(i, j) for j in range(len(vocab))
                ] 
                for i in range(len(vocab)) 
            ]

pmi_df = pd.DataFrame(pmi_metric, index = vocab, columns = vocab)

In [50]:
pmi_df.style.background_gradient(cmap='Blues')

,가,기분,나쁘다,날씨,내일,너무,매우,오늘,이,정말,조금,좋다,흐리다
가,0.000000,0.000000,0.000000,1.187627,1.772590,1.772590,0.000000,0.772590,0.000000,0.000000,1.772590,0.000000,0.000000
기분,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.772590,2.187627,0.000000,0.000000,0.000000,0.000000
나쁘다,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.357552,0.000000,1.772590,0.000000,0.000000,0.000000,0.000000
날씨,1.772590,0.000000,0.000000,0.000000,2.357552,0.000000,0.000000,1.357552,0.000000,0.000000,0.000000,0.000000,0.000000
내일,0.000000,0.000000,0.000000,2.772590,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
너무,1.187627,0.000000,0.000000,1.187627,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.772590,0.000000
매우,0.000000,1.450661,3.772590,0.000000,0.000000,0.000000,0.000000,0.000000,1.187627,0.000000,0.000000,0.000000,0.000000
오늘,0.000000,2.035624,0.000000,1.772590,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
이,0.000000,1.713696,0.000000,0.000000,0.000000,0.000000,2.035624,1.035624,0.000000,2.035624,0.000000,0.000000,0.000000
정말,0.000000,1.450661,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.187627,0.000000,0.000000,2.772590,0.000000
